# Notebook 00 — Environment, merge, and source audit

This notebook is the single owner of the two lossless source merges used by the
research workflow. It reads the original CSV files, preserves every source row,
adds deterministic provenance, and writes canonical **CSV** artifacts for later
notebooks.

The notebook deliberately does not clean, impute, deduplicate, cap, or remove
high values. Its responsibilities are reproducibility, source-level auditing,
lossless merging, and read-back validation. Scientific preprocessing remains in
Notebook 01.

In [ ]:
import hashlib
import json
import os
import platform
import random
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "00_environment_and_data_audit"

for directory in [INTERIM_DATA_DIR, REPORT_TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not (PROJECT_ROOT / "data").is_dir() or not (PROJECT_ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {PROJECT_ROOT}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

environment_table = pd.DataFrame(
    {
        "item": [
            "executed_at_utc", "python_executable", "python_version",
            "platform", "pandas_version", "numpy_version", "seed",
            "project_root", "working_directory",
        ],
        "value": [
            datetime.now(timezone.utc).isoformat(), sys.executable,
            sys.version.split()[0], platform.platform(), pd.__version__,
            np.__version__, SEED, str(PROJECT_ROOT), str(CURRENT_DIR),
        ],
    }
)
display(environment_table)

## 1. Reproducible merge contract

`MERGE_EXECUTION_MODE = "rebuild"` makes this execution recreate both merged
datasets from the immutable raw CSVs. Each output is first written to a sibling
`.tmp.csv` file and is moved into place only after all source chunks succeed.
Therefore an interrupted run cannot present a partial file as a completed merge.

All measurement fields remain lossless strings at this stage. Their numeric and
datetime interpretations are audited, but conversion and treatment decisions are
deferred to preprocessing.

In [ ]:
MERGE_EXECUTION_MODE = "rebuild"
FIFTEEN_MINUTE_CHUNK_SIZE = 250_000
SIX_HOUR_CHUNK_SIZE = 100_000

FIFTEEN_MINUTE_SOURCE_FILES = sorted(RAW_DATA_DIR.glob("smart_15min_*.csv"))
SIX_HOUR_SOURCE_FILES = sorted(RAW_DATA_DIR.glob("smart_6hour_*.csv"))

if len(FIFTEEN_MINUTE_SOURCE_FILES) != 3:
    raise FileNotFoundError(
        f"Expected three 15-minute CSV files, found {len(FIFTEEN_MINUTE_SOURCE_FILES)}"
    )
if len(SIX_HOUR_SOURCE_FILES) != 5:
    raise FileNotFoundError(
        f"Expected five 6-hour CSV files, found {len(SIX_HOUR_SOURCE_FILES)}"
    )

FIFTEEN_MINUTE_MERGED_PATH = INTERIM_DATA_DIR / "01_merged_interval_data.csv"
SIX_HOUR_MERGED_PATH = INTERIM_DATA_DIR / "01_merged_6hour_interval_data.csv"
MERGE_MANIFEST_PATH = INTERIM_DATA_DIR / "00_merge_manifest.json"
DTYPE_CONTRACT_PATH = INTERIM_DATA_DIR / "00_csv_dtype_contract.json"

FIFTEEN_MINUTE_EXPECTED_RAW_COLUMNS = [
    "household_ID", "timestamp", "date", "time", "obis",
    "importkwh(kwh)", "exportkwh(kwh)", "importkvarh(kvarh)",
    "exportkvarh(kvarh)", "powerfactor", "avgcurrent(v)",
    "avgexportkw(kw)", "avgimportkw(kw)", "avgvoltage(v)",
    "phaseainstcurrent(a)", "phaseainstvoltage(v)",
    "phasebinstcurrent(a)", "phasebinstvoltage(v)",
    "phasecinstcurrent(a)", "phasecinstvoltage(v)",
]

FIFTEEN_MINUTE_RENAME = {
    "household_ID": "household_id",
    "avgimportkw(kw)": "avg_import_kw",
    "avgexportkw(kw)": "avg_export_kw",
    "importkwh(kwh)": "import_kwh",
    "exportkwh(kwh)": "export_kwh",
    "importkvarh(kvarh)": "import_kvarh",
    "exportkvarh(kvarh)": "export_kvarh",
    "powerfactor": "power_factor",
    "avgcurrent(v)": "avg_current",
    "avgvoltage(v)": "avg_voltage",
    "phaseainstcurrent(a)": "phase_a_inst_current",
    "phaseainstvoltage(v)": "phase_a_inst_voltage",
    "phasebinstcurrent(a)": "phase_b_inst_current",
    "phasebinstvoltage(v)": "phase_b_inst_voltage",
    "phasecinstcurrent(a)": "phase_c_inst_current",
    "phasecinstvoltage(v)": "phase_c_inst_voltage",
}

FIFTEEN_MINUTE_CANONICAL_COLUMNS = [
    "household_id", "timestamp", "date", "time", "obis",
    "import_kwh", "export_kwh", "import_kvarh", "export_kvarh",
    "power_factor", "avg_current", "avg_export_kw", "avg_import_kw",
    "avg_voltage", "phase_a_inst_current", "phase_a_inst_voltage",
    "phase_b_inst_current", "phase_b_inst_voltage",
    "phase_c_inst_current", "phase_c_inst_voltage",
    "source_file", "source_row_number",
]

SIX_HOUR_EXPECTED_RAW_COLUMNS = [
    "household_ID", "BRANCH", "DEPOT", "substation_ID",
    "SUB_TYPE", "E_BILLING", "EXPORT", "DATE", "TIME",
    "TOTAL_IMPORT (kWh)", "TOTAL_EXPORT (kWh)",
    "TOTAL_IMPORT - PV1 (kWh)", "TOTAL_EXPORT - PV1 (kWh)",
    "TR1_TOTAL_IMPORT (kWh)", "TR2_TOTAL_IMPORT (kWh)",
    "TR3_TOTAL_IMPORT (kWh)", "TR1_TOTAL_EXPORT (kWh)",
    "TR2_TOTAL_EXPORT (kWh)", "TR3_TOTAL_EXPORT (kWh)",
    "TR1_TOTAL_IMPORT - PV1 (kWh)", "TR2_TOTAL_IMPORT - PV1 (kWh)",
    "TR3_TOTAL_IMPORT - PV1 (kWh)", "TR1_TOTAL_EXPORT - PV1 (kWh)",
    "TR2_TOTAL_EXPORT - PV1 (kWh)", "TR3_TOTAL_EXPORT - PV1 (kWh)",
    "PHASE_A_CURRENT (A)", "PHASE_A_VOLTAGE (V)", "FREQUENCY (Hz)",
    "PHASE_B_CURRENT (A)", "PHASE_B_VOLTAGE (V)",
    "PHASE_C_CURRENT (A)", "PHASE_C_VOLTAGE (V)",
]

SIX_HOUR_RENAME = {
    "household_ID": "household_id", "BRANCH": "branch", "DEPOT": "depot",
    "substation_ID": "substation_id", "SUB_TYPE": "meter_sub_type",
    "E_BILLING": "e_billing", "EXPORT": "export_enabled",
    "DATE": "date", "TIME": "time",
    "TOTAL_IMPORT (kWh)": "total_import_kwh",
    "TOTAL_EXPORT (kWh)": "total_export_kwh",
    "TOTAL_IMPORT - PV1 (kWh)": "total_import_pv1_kwh",
    "TOTAL_EXPORT - PV1 (kWh)": "total_export_pv1_kwh",
    "TR1_TOTAL_IMPORT (kWh)": "tr1_total_import_kwh",
    "TR2_TOTAL_IMPORT (kWh)": "tr2_total_import_kwh",
    "TR3_TOTAL_IMPORT (kWh)": "tr3_total_import_kwh",
    "TR1_TOTAL_EXPORT (kWh)": "tr1_total_export_kwh",
    "TR2_TOTAL_EXPORT (kWh)": "tr2_total_export_kwh",
    "TR3_TOTAL_EXPORT (kWh)": "tr3_total_export_kwh",
    "TR1_TOTAL_IMPORT - PV1 (kWh)": "tr1_total_import_pv1_kwh",
    "TR2_TOTAL_IMPORT - PV1 (kWh)": "tr2_total_import_pv1_kwh",
    "TR3_TOTAL_IMPORT - PV1 (kWh)": "tr3_total_import_pv1_kwh",
    "TR1_TOTAL_EXPORT - PV1 (kWh)": "tr1_total_export_pv1_kwh",
    "TR2_TOTAL_EXPORT - PV1 (kWh)": "tr2_total_export_pv1_kwh",
    "TR3_TOTAL_EXPORT - PV1 (kWh)": "tr3_total_export_pv1_kwh",
    "PHASE_A_CURRENT (A)": "phase_a_current",
    "PHASE_A_VOLTAGE (V)": "phase_a_voltage",
    "FREQUENCY (Hz)": "frequency_hz",
    "PHASE_B_CURRENT (A)": "phase_b_current",
    "PHASE_B_VOLTAGE (V)": "phase_b_voltage",
    "PHASE_C_CURRENT (A)": "phase_c_current",
    "PHASE_C_VOLTAGE (V)": "phase_c_voltage",
}

SIX_HOUR_CANONICAL_COLUMNS = [
    "household_id", "branch", "depot", "substation_id",
    "meter_sub_type", "e_billing", "export_enabled", "date", "time",
    "total_import_kwh", "total_export_kwh", "total_import_pv1_kwh",
    "total_export_pv1_kwh", "tr1_total_import_kwh",
    "tr2_total_import_kwh", "tr3_total_import_kwh",
    "tr1_total_export_kwh", "tr2_total_export_kwh",
    "tr3_total_export_kwh", "tr1_total_import_pv1_kwh",
    "tr2_total_import_pv1_kwh", "tr3_total_import_pv1_kwh",
    "tr1_total_export_pv1_kwh", "tr2_total_export_pv1_kwh",
    "tr3_total_export_pv1_kwh", "phase_a_current", "phase_a_voltage",
    "frequency_hz", "phase_b_current", "phase_b_voltage",
    "phase_c_current", "phase_c_voltage", "source_file",
    "source_row_number", "record_frequency",
]

SIX_HOUR_NUMERIC_COLUMNS = [
    column for column in SIX_HOUR_CANONICAL_COLUMNS
    if column.endswith("_kwh") or column in {
        "phase_a_current", "phase_a_voltage", "frequency_hz",
        "phase_b_current", "phase_b_voltage", "phase_c_current", "phase_c_voltage",
    }
]

CSV_DTYPE_CONTRACT = {
    "15_minute": {
        column: ("integer" if column == "source_row_number" else "string")
        for column in FIFTEEN_MINUTE_CANONICAL_COLUMNS
    },
    "6_hour": {
        column: ("integer" if column == "source_row_number" else "string")
        for column in SIX_HOUR_CANONICAL_COLUMNS
    },
}

print(f"Execution mode: {MERGE_EXECUTION_MODE}")
print(f"15-minute output: {FIFTEEN_MINUTE_MERGED_PATH}")
print(f"6-hour output: {SIX_HOUR_MERGED_PATH}")

## 2. Raw-file inventory and schema validation

This is a fail-fast gate. A missing, additional, or reordered raw column is
reported before any merged artifact is replaced.

In [ ]:
def inspect_source_headers(source_files, expected_columns, dataset_name):
    rows = []
    for source_path in source_files:
        actual_columns = list(pd.read_csv(source_path, nrows=0).columns)
        rows.append(
            {
                "dataset": dataset_name,
                "source_file": source_path.name,
                "size_bytes": source_path.stat().st_size,
                "size_gb": source_path.stat().st_size / (1024 ** 3),
                "column_count": len(actual_columns),
                "missing_columns": sorted(set(expected_columns) - set(actual_columns)),
                "additional_columns": sorted(set(actual_columns) - set(expected_columns)),
                "column_order_matches": actual_columns == expected_columns,
                "modified_utc": datetime.fromtimestamp(
                    source_path.stat().st_mtime, tz=timezone.utc
                ).isoformat(),
            }
        )
    result = pd.DataFrame(rows)
    invalid = (
        result["missing_columns"].map(bool)
        | result["additional_columns"].map(bool)
        | ~result["column_order_matches"]
    )
    if invalid.any():
        display(result)
        raise ValueError(f"{dataset_name} raw CSV schema contract failed")
    return result

fifteen_minute_header_audit = inspect_source_headers(
    FIFTEEN_MINUTE_SOURCE_FILES,
    FIFTEEN_MINUTE_EXPECTED_RAW_COLUMNS,
    "15_minute",
)
six_hour_header_audit = inspect_source_headers(
    SIX_HOUR_SOURCE_FILES,
    SIX_HOUR_EXPECTED_RAW_COLUMNS,
    "6_hour",
)
source_header_audit = pd.concat(
    [fifteen_minute_header_audit, six_hour_header_audit], ignore_index=True
)
display(source_header_audit)

## 3. Chunked atomic CSV merge helpers

`source_row_number` is one-based within each raw file. Together with
`source_file`, it forms a deterministic source provenance key. The 6-hour merge
additionally receives `record_frequency = "6hour"`.

In [ ]:
def count_source_rows(source_files, key_column, chunk_size):
    rows = []
    for source_path in source_files:
        row_count = sum(
            len(chunk)
            for chunk in pd.read_csv(
                source_path,
                usecols=[key_column],
                dtype="string",
                chunksize=chunk_size,
                low_memory=False,
            )
        )
        rows.append({"source_file": source_path.name, "rows_written": row_count})
    return pd.DataFrame(rows)


def merge_sources_to_csv(
    source_files,
    output_path,
    rename_map,
    canonical_columns,
    chunk_size,
    dataset_name,
    record_frequency=None,
):
    temp_path = output_path.with_name(f"{output_path.stem}.tmp.csv")

    if MERGE_EXECUTION_MODE == "validate_existing" and output_path.exists():
        print(f"Validating existing {dataset_name} CSV: {output_path}")
        merge_log = count_source_rows(source_files, "household_ID", chunk_size)
        return merge_log, None

    if MERGE_EXECUTION_MODE != "rebuild":
        raise ValueError(
            "MERGE_EXECUTION_MODE must be 'rebuild' or 'validate_existing'"
        )

    if temp_path.exists():
        temp_path.unlink()

    merge_rows = []
    source_audit_rows = []
    audit = {
        "household_ids": set(),
        "record_type_counts": Counter(),
        "key_missing_counts": Counter(),
        "value_stats": Counter(),
        "value_min": None,
        "value_max": None,
        "value_samples": [],
        "within_chunk_duplicate_rows": Counter(),
        "within_chunk_duplicate_groups": Counter(),
        "lp_household_day_counts": Counter(),
        "lp_avgimportkw_counts": Counter(),
    }
    first_write = True

    try:
        for source_path in source_files:
            source_rows = 0
            source_households = set()
            source_invalid_dates = 0
            source_min_date = None
            source_max_date = None

            for chunk in pd.read_csv(
                source_path,
                chunksize=chunk_size,
                dtype="string",
                low_memory=False,
            ):
                source_row_start = source_rows + 1
                source_rows += len(chunk)

                household = chunk["household_ID"].astype("string").str.strip()
                source_households.update(household.dropna().unique().tolist())
                audit["household_ids"].update(household.dropna().unique().tolist())

                date_column = "date" if "date" in chunk.columns else "DATE"
                parsed_dates = pd.to_datetime(chunk[date_column], errors="coerce")
                source_invalid_dates += int(parsed_dates.isna().sum())
                if parsed_dates.notna().any():
                    chunk_min = parsed_dates.min()
                    chunk_max = parsed_dates.max()
                    source_min_date = chunk_min if source_min_date is None else min(source_min_date, chunk_min)
                    source_max_date = chunk_max if source_max_date is None else max(source_max_date, chunk_max)

                if dataset_name == "15_minute":
                    key_columns = [
                        "household_ID", "timestamp", "date", "time", "obis",
                        "avgimportkw(kw)",
                    ]
                    audit["key_missing_counts"].update(
                        chunk[key_columns].isna().sum().astype("int64").to_dict()
                    )

                    obis = chunk["obis"].astype("string").str.strip().str.upper()
                    audit["record_type_counts"].update(
                        obis.fillna("<MISSING>").astype(str)
                    )

                    raw_import = chunk["avgimportkw(kw)"]
                    import_kw = pd.to_numeric(raw_import, errors="coerce")
                    audit["value_stats"]["rows"] += len(chunk)
                    audit["value_stats"]["missing"] += int(raw_import.isna().sum())
                    audit["value_stats"]["non_numeric"] += int(
                        (raw_import.notna() & import_kw.isna()).sum()
                    )
                    audit["value_stats"]["negative"] += int(import_kw.lt(0).sum())
                    audit["value_stats"]["zero"] += int(import_kw.eq(0).sum())
                    if import_kw.notna().any():
                        chunk_minimum = float(import_kw.min())
                        chunk_maximum = float(import_kw.max())
                        audit["value_min"] = chunk_minimum if audit["value_min"] is None else min(audit["value_min"], chunk_minimum)
                        audit["value_max"] = chunk_maximum if audit["value_max"] is None else max(audit["value_max"], chunk_maximum)
                        audit["value_samples"].append(import_kw.iloc[::50].dropna().astype("float64"))

                    lp_mask = obis.eq("LP").fillna(False)
                    lp_import_numeric = import_kw.notna()
                    lp_import_finite = lp_import_numeric & np.isfinite(import_kw)
                    lp_import_missing = lp_mask & raw_import.isna()
                    lp_import_non_numeric = lp_mask & raw_import.notna() & import_kw.isna()
                    lp_import_non_finite = lp_mask & lp_import_numeric & ~np.isfinite(import_kw)
                    lp_import_negative = lp_mask & lp_import_finite & import_kw.lt(0)
                    lp_import_valid = lp_mask & lp_import_finite & import_kw.ge(0)
                    audit["lp_avgimportkw_counts"]["lp_rows"] += int(lp_mask.sum())
                    audit["lp_avgimportkw_counts"]["missing"] += int(lp_import_missing.sum())
                    audit["lp_avgimportkw_counts"]["non_numeric"] += int(lp_import_non_numeric.sum())
                    audit["lp_avgimportkw_counts"]["non_finite"] += int(lp_import_non_finite.sum())
                    audit["lp_avgimportkw_counts"]["negative"] += int(lp_import_negative.sum())
                    audit["lp_avgimportkw_counts"]["valid_finite_nonnegative"] += int(lp_import_valid.sum())

                    lp_keys = chunk.loc[
                        lp_mask, ["household_ID", "date", "time", "obis"]
                    ].dropna(subset=["household_ID", "date", "time"])
                    duplicate_mask = lp_keys.duplicated(
                        ["household_ID", "date", "time", "obis"], keep=False
                    )
                    audit["within_chunk_duplicate_rows"][source_path.name] += int(duplicate_mask.sum())
                    audit["within_chunk_duplicate_groups"][source_path.name] += int(
                        lp_keys.loc[duplicate_mask].drop_duplicates().shape[0]
                    )

                    lp_days = chunk.loc[lp_mask, ["household_ID", "date"]].dropna()
                    if not lp_days.empty:
                        grouped = lp_days.groupby(
                            ["household_ID", "date"], sort=False
                        ).size()
                        audit["lp_household_day_counts"].update(
                            {key: int(value) for key, value in grouped.items()}
                        )

                merged = chunk.rename(columns=rename_map)
                merged["source_file"] = source_path.name
                merged["source_row_number"] = np.arange(
                    source_row_start,
                    source_row_start + len(merged),
                    dtype=np.int64,
                )
                if record_frequency is not None:
                    merged["record_frequency"] = record_frequency
                merged = merged[canonical_columns]
                merged.to_csv(
                    temp_path,
                    mode="w" if first_write else "a",
                    header=first_write,
                    index=False,
                    encoding="utf-8",
                    lineterminator="\n",
                )
                first_write = False

            merge_rows.append(
                {"dataset": dataset_name, "source_file": source_path.name, "rows_written": source_rows}
            )
            source_audit_rows.append(
                {
                    "dataset": dataset_name,
                    "source_file": source_path.name,
                    "rows": source_rows,
                    "unique_households": len(source_households),
                    "invalid_dates": source_invalid_dates,
                    "min_date": source_min_date,
                    "max_date": source_max_date,
                }
            )

        if first_write:
            raise RuntimeError(f"No rows were written for {dataset_name}")
        os.replace(temp_path, output_path)
    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise

    print(f"Created {dataset_name} CSV: {output_path}")
    return pd.DataFrame(merge_rows), {
        "source_audit": pd.DataFrame(source_audit_rows),
        **audit,
    }


def sha256_file(path, block_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

## 4. Build the canonical 15-minute CSV

The following source audit is gathered during the same pass that writes the
merge, avoiding repeated full scans. High and extreme demand values are retained.
The duplicate screen shown here is deliberately limited to duplicates found
inside a read chunk; the complete cross-file duplicate audit belongs to Notebook
01 and must be completed before cleaning decisions are applied.

In [ ]:
fifteen_minute_merge_log, fifteen_minute_raw_audit = merge_sources_to_csv(
    source_files=FIFTEEN_MINUTE_SOURCE_FILES,
    output_path=FIFTEEN_MINUTE_MERGED_PATH,
    rename_map=FIFTEEN_MINUTE_RENAME,
    canonical_columns=FIFTEEN_MINUTE_CANONICAL_COLUMNS,
    chunk_size=FIFTEEN_MINUTE_CHUNK_SIZE,
    dataset_name="15_minute",
)

display(fifteen_minute_merge_log)

if fifteen_minute_raw_audit is not None:
    fifteen_minute_source_audit = fifteen_minute_raw_audit["source_audit"]
    display(fifteen_minute_source_audit)

    fifteen_minute_record_types = pd.DataFrame(
        fifteen_minute_raw_audit["record_type_counts"].items(),
        columns=["obis", "rows"],
    ).sort_values("rows", ascending=False)
    fifteen_minute_record_types["percentage"] = (
        100 * fifteen_minute_record_types["rows"] / fifteen_minute_record_types["rows"].sum()
    )

    fifteen_minute_missing = pd.DataFrame(
        fifteen_minute_raw_audit["key_missing_counts"].items(),
        columns=["raw_column", "missing_count"],
    )
    fifteen_minute_missing["missing_percentage"] = (
        100 * fifteen_minute_missing["missing_count"] / fifteen_minute_merge_log["rows_written"].sum()
    )

    value_stats = dict(fifteen_minute_raw_audit["value_stats"])
    value_stats["minimum"] = fifteen_minute_raw_audit["value_min"]
    value_stats["maximum"] = fifteen_minute_raw_audit["value_max"]
    fifteen_minute_value_summary = pd.DataFrame([value_stats])

    sampled_values = pd.concat(
        fifteen_minute_raw_audit["value_samples"], ignore_index=True
    )
    fifteen_minute_value_quantiles = sampled_values.quantile(
        [0, 0.01, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999, 1.00]
    ).rename("estimated_avg_import_kw")

    fifteen_minute_duplicate_screen = pd.DataFrame(
        {
            "source_file": [path.name for path in FIFTEEN_MINUTE_SOURCE_FILES],
            "duplicate_LP_rows_within_chunk": [
                fifteen_minute_raw_audit["within_chunk_duplicate_rows"][path.name]
                for path in FIFTEEN_MINUTE_SOURCE_FILES
            ],
            "duplicate_LP_key_groups_within_chunk": [
                fifteen_minute_raw_audit["within_chunk_duplicate_groups"][path.name]
                for path in FIFTEEN_MINUTE_SOURCE_FILES
            ],
        }
    )

    lp_counts = pd.Series(
        fifteen_minute_raw_audit["lp_household_day_counts"], dtype="int64"
    )
    fifteen_minute_coverage_distribution = (
        lp_counts.value_counts().sort_index().rename_axis("observed_intervals").rename("household_days").reset_index()
    )
    fifteen_minute_coverage_summary = pd.DataFrame(
        {
            "measure": [
                "household_days", "exactly_96_intervals", "fewer_than_96",
                "more_than_96", "near_complete_70_to_95",
            ],
            "value": [
                len(lp_counts), int(lp_counts.eq(96).sum()), int(lp_counts.lt(96).sum()),
                int(lp_counts.gt(96).sum()), int(lp_counts.between(70, 95).sum()),
            ],
        }
    )

    display(fifteen_minute_record_types)
    display(fifteen_minute_missing)
    display(fifteen_minute_value_summary)
    display(fifteen_minute_value_quantiles.to_frame())
    display(fifteen_minute_duplicate_screen)
    display(fifteen_minute_coverage_summary)
    display(fifteen_minute_coverage_distribution.head(20))
    print("No demand value was removed, clipped, winsorized, or replaced in this notebook.")

## 5. Build the canonical 6-hour CSV

This source remains a separate dataset. It is not joined to 15-minute readings in
this notebook because the later metadata join must respect time validity.

In [ ]:
six_hour_merge_log, six_hour_raw_audit = merge_sources_to_csv(
    source_files=SIX_HOUR_SOURCE_FILES,
    output_path=SIX_HOUR_MERGED_PATH,
    rename_map=SIX_HOUR_RENAME,
    canonical_columns=SIX_HOUR_CANONICAL_COLUMNS,
    chunk_size=SIX_HOUR_CHUNK_SIZE,
    dataset_name="6_hour",
    record_frequency="6hour",
)

display(six_hour_merge_log)
if six_hour_raw_audit is not None:
    six_hour_source_audit = six_hour_raw_audit["source_audit"]
    display(six_hour_source_audit)

## 6. Independent read-back validation

Validation reopens the completed CSVs rather than trusting in-memory merge
counters. For every source block it verifies the exact `1..N` row-number sequence.
Combined with source-row reconciliation, this proves uniqueness and completeness
of the `(source_file, source_row_number)` provenance key without storing tens of
millions of keys in memory.

In [ ]:
def validate_merged_csv(
    output_path,
    canonical_columns,
    source_files,
    merge_log,
    chunk_size,
    dataset_name,
):
    header = list(pd.read_csv(output_path, nrows=0).columns)
    read_columns = [
        "household_id", "date", "time", "source_file", "source_row_number"
    ]
    source_counts = Counter()
    source_next = defaultdict(lambda: 1)
    households = set()
    sequence_errors = []
    repeated_header_rows = 0
    missing_provenance_rows = 0
    total_rows = 0
    first_samples = {}
    last_samples = {}
    minimum_date = None
    maximum_date = None

    for chunk in pd.read_csv(
        output_path,
        usecols=read_columns,
        dtype="string",
        chunksize=chunk_size,
        low_memory=False,
    ):
        total_rows += len(chunk)
        repeated_header_rows += int(
            chunk["source_file"].eq("source_file").fillna(False).sum()
            + chunk["household_id"].eq("household_id").fillna(False).sum()
        )
        missing_provenance_rows += int(
            chunk[["source_file", "source_row_number"]].isna().any(axis=1).sum()
        )
        households.update(chunk["household_id"].dropna().astype(str).unique())
        source_counts.update(chunk["source_file"].dropna().astype(str))

        parsed_dates = pd.to_datetime(chunk["date"], errors="coerce")
        if parsed_dates.notna().any():
            chunk_min = parsed_dates.min()
            chunk_max = parsed_dates.max()
            minimum_date = chunk_min if minimum_date is None else min(minimum_date, chunk_min)
            maximum_date = chunk_max if maximum_date is None else max(maximum_date, chunk_max)

        for source_file, group in chunk.groupby("source_file", sort=False, dropna=False):
            if pd.isna(source_file):
                continue
            source_file = str(source_file)
            numbers = pd.to_numeric(group["source_row_number"], errors="coerce").to_numpy()
            expected = np.arange(
                source_next[source_file], source_next[source_file] + len(group), dtype=np.int64
            )
            if np.isnan(numbers).any() or not np.array_equal(numbers, expected):
                sequence_errors.append(
                    {
                        "source_file": source_file,
                        "expected_start": int(expected[0]),
                        "observed_start": None if np.isnan(numbers[0]) else int(numbers[0]),
                    }
                )
            source_next[source_file] += len(group)
            sample_columns = [
                "household_id", "date", "time", "source_file", "source_row_number"
            ]
            if source_file not in first_samples:
                first_samples[source_file] = group.iloc[0][sample_columns].to_dict()
            last_samples[source_file] = group.iloc[-1][sample_columns].to_dict()

    expected_counts = merge_log.set_index("source_file")["rows_written"].astype("int64")
    reconciliation_rows = []
    for source_path in source_files:
        source_file = source_path.name
        expected_rows = int(expected_counts[source_file])
        observed_rows = int(source_counts[source_file])
        reconciliation_rows.append(
            {
                "dataset": dataset_name,
                "source_file": source_file,
                "expected_rows": expected_rows,
                "merged_rows": observed_rows,
                "row_difference": observed_rows - expected_rows,
                "final_source_row_number": source_next[source_file] - 1,
            }
        )
    reconciliation = pd.DataFrame(reconciliation_rows)

    samples = []
    for source_path in source_files:
        source_file = source_path.name
        samples.append({"sample_position": "first", **first_samples[source_file]})
        samples.append({"sample_position": "last", **last_samples[source_file]})
    sample_table = pd.DataFrame(samples)

    expected_total = int(expected_counts.sum())
    expected_sources = {path.name for path in source_files}
    observed_sources = set(source_counts)
    checks = [
        ("output_exists", output_path.exists()),
        ("output_is_csv", output_path.suffix.lower() == ".csv"),
        ("header_order_matches_contract", header == canonical_columns),
        ("total_rows_match_sources", total_rows == expected_total),
        ("all_source_counts_reconcile", reconciliation["row_difference"].eq(0).all()),
        ("source_sets_match", observed_sources == expected_sources),
        ("provenance_has_no_missing_values", missing_provenance_rows == 0),
        ("no_repeated_header_rows", repeated_header_rows == 0),
        ("provenance_sequence_is_exact", len(sequence_errors) == 0),
        ("provenance_key_is_unique", len(sequence_errors) == 0 and reconciliation["row_difference"].eq(0).all()),
    ]
    validation = pd.DataFrame(
        {
            "dataset": dataset_name,
            "check": [item[0] for item in checks],
            "passed": [bool(item[1]) for item in checks],
        }
    )

    if not validation["passed"].all():
        display(validation)
        display(reconciliation)
        raise AssertionError(f"{dataset_name} merged CSV validation failed")

    signature = {
        "dataset": dataset_name,
        "path": str(output_path.relative_to(PROJECT_ROOT)),
        "format": "csv",
        "rows": total_rows,
        "columns": len(header),
        "size_bytes": output_path.stat().st_size,
        "sha256": sha256_file(output_path),
        "unique_households": len(households),
        "first_date": None if minimum_date is None else str(minimum_date.date()),
        "last_date": None if maximum_date is None else str(maximum_date.date()),
    }
    return validation, reconciliation, sample_table, signature, households


fifteen_validation, fifteen_reconciliation, fifteen_samples, fifteen_signature, fifteen_households = validate_merged_csv(
    FIFTEEN_MINUTE_MERGED_PATH,
    FIFTEEN_MINUTE_CANONICAL_COLUMNS,
    FIFTEEN_MINUTE_SOURCE_FILES,
    fifteen_minute_merge_log,
    FIFTEEN_MINUTE_CHUNK_SIZE,
    "15_minute",
)
display(fifteen_validation)
display(fifteen_reconciliation)
display(fifteen_samples)
display(pd.DataFrame([fifteen_signature]))

In [ ]:
six_validation, six_reconciliation, six_samples, six_signature, six_households = validate_merged_csv(
    SIX_HOUR_MERGED_PATH,
    SIX_HOUR_CANONICAL_COLUMNS,
    SIX_HOUR_SOURCE_FILES,
    six_hour_merge_log,
    SIX_HOUR_CHUNK_SIZE,
    "6_hour",
)
display(six_validation)
display(six_reconciliation)
display(six_samples)
display(pd.DataFrame([six_signature]))

## 7. Six-hour source-quality and time-validity audit

This audit does not flatten network metadata to one value per household. It
counts households whose branch, depot, substation, or meter attributes change so
that Notebook 01 can perform an as-of, time-valid join. Cumulative-register
decreases are flags for later investigation, not values to remove automatically.

In [ ]:
six_missing_counts = pd.Series(0, index=SIX_HOUR_CANONICAL_COLUMNS, dtype="int64")
numeric_state = {
    column: {"missing_or_unparseable": 0, "invalid_text": 0, "negative": 0, "minimum": None, "maximum": None}
    for column in SIX_HOUR_NUMERIC_COLUMNS
}
metadata_columns = [
    "branch", "depot", "substation_id", "meter_sub_type", "e_billing", "export_enabled"
]
household_metadata = defaultdict(lambda: {column: set() for column in metadata_columns})
duplicate_hash_parts = []
coverage_hash_parts = []
invalid_timestamps = 0
timestamp_min = None
timestamp_max = None
total_six_rows = 0

audit_columns = SIX_HOUR_CANONICAL_COLUMNS
for chunk in pd.read_csv(
    SIX_HOUR_MERGED_PATH,
    usecols=audit_columns,
    dtype="string",
    chunksize=SIX_HOUR_CHUNK_SIZE,
    low_memory=False,
):
    total_six_rows += len(chunk)
    six_missing_counts = six_missing_counts.add(chunk.isna().sum(), fill_value=0)

    timestamp_text = chunk["date"].astype("string") + " " + chunk["time"].astype("string")
    timestamps = pd.to_datetime(timestamp_text, errors="coerce")
    invalid_timestamps += int(timestamps.isna().sum())
    if timestamps.notna().any():
        timestamp_min = timestamps.min() if timestamp_min is None else min(timestamp_min, timestamps.min())
        timestamp_max = timestamps.max() if timestamp_max is None else max(timestamp_max, timestamps.max())

    duplicate_hash_parts.append(
        pd.util.hash_pandas_object(
            chunk[["household_id", "date", "time"]].fillna("<MISSING>"), index=False
        ).to_numpy(dtype="uint64")
    )
    coverage_hash_parts.append(
        pd.util.hash_pandas_object(
            chunk[["household_id", "date"]].fillna("<MISSING>"), index=False
        ).to_numpy(dtype="uint64")
    )

    for column in SIX_HOUR_NUMERIC_COLUMNS:
        values = pd.to_numeric(chunk[column], errors="coerce")
        state = numeric_state[column]
        state["missing_or_unparseable"] += int(values.isna().sum())
        state["invalid_text"] += int((chunk[column].notna() & values.isna()).sum())
        state["negative"] += int(values.lt(0).sum())
        if values.notna().any():
            local_min = float(values.min())
            local_max = float(values.max())
            state["minimum"] = local_min if state["minimum"] is None else min(state["minimum"], local_min)
            state["maximum"] = local_max if state["maximum"] is None else max(state["maximum"], local_max)

    for household_id, group in chunk.groupby("household_id", dropna=False):
        if pd.isna(household_id):
            continue
        household_id = str(household_id)
        for column in metadata_columns:
            household_metadata[household_id][column].update(
                group[column].dropna().astype(str).unique()
            )

duplicate_hashes = np.concatenate(duplicate_hash_parts)
duplicate_hashes.sort()
duplicate_row_mask = np.r_[False, duplicate_hashes[1:] == duplicate_hashes[:-1]]
duplicate_key_groups = int(
    np.sum(
        (duplicate_hashes[1:] == duplicate_hashes[:-1])
        & np.r_[True, duplicate_hashes[1:-1] != duplicate_hashes[:-2]]
    )
) if len(duplicate_hashes) > 1 else 0
duplicate_rows_beyond_first = int(duplicate_row_mask.sum())

coverage_hashes = np.concatenate(coverage_hash_parts)
coverage_hashes.sort()
_, coverage_counts = np.unique(coverage_hashes, return_counts=True)
six_hour_coverage_summary = pd.DataFrame(
    {
        "measure": [
            "household_days", "complete_4_rows", "fewer_than_4_rows", "more_than_4_rows"
        ],
        "value": [
            len(coverage_counts), int(np.sum(coverage_counts == 4)),
            int(np.sum(coverage_counts < 4)), int(np.sum(coverage_counts > 4)),
        ],
    }
)

six_hour_missing_table = six_missing_counts.rename_axis("column").reset_index(name="missing_count")
six_hour_missing_table["missing_percentage"] = 100 * six_hour_missing_table["missing_count"] / total_six_rows
six_hour_numeric_table = pd.DataFrame(
    [{"column": column, **state} for column, state in numeric_state.items()]
)
metadata_stability_table = pd.DataFrame(
    [
        {
            "column": column,
            "households": len(household_metadata),
            "stable_households": sum(len(values[column]) == 1 for values in household_metadata.values()),
            "changing_households": sum(len(values[column]) > 1 for values in household_metadata.values()),
            "maximum_values_per_household": max(len(values[column]) for values in household_metadata.values()),
        }
        for column in metadata_columns
    ]
)
six_hour_time_and_duplicate_summary = pd.DataFrame(
    {
        "measure": [
            "rows", "invalid_timestamps", "first_timestamp", "last_timestamp",
            "duplicate_key_groups_by_64bit_hash", "duplicate_rows_beyond_first_by_64bit_hash",
        ],
        "value": [
            total_six_rows, invalid_timestamps, timestamp_min, timestamp_max,
            duplicate_key_groups, duplicate_rows_beyond_first,
        ],
    }
)

display(six_hour_time_and_duplicate_summary)
display(six_hour_coverage_summary)
display(six_hour_missing_table.sort_values("missing_count", ascending=False))
display(six_hour_numeric_table)
display(metadata_stability_table)
print("Duplicate keys are screened with deterministic pandas 64-bit hashes; matching hashes are treated as candidate duplicate keys.")

## 8. Cross-dataset reconciliation, reports, and manifest

The household comparison checks population coverage only. It does not authorize a
timeless metadata join. The final manifest records row counts, byte sizes,
SHA-256 content signatures, column contracts, and validation results.

In [ ]:
household_reconciliation = pd.DataFrame(
    {
        "measure": [
            "15_minute_households", "6_hour_households", "shared_households",
            "15_minute_only", "6_hour_only",
        ],
        "value": [
            len(fifteen_households), len(six_households),
            len(fifteen_households & six_households),
            len(fifteen_households - six_households),
            len(six_households - fifteen_households),
        ],
    }
)
display(household_reconciliation)


def atomic_dataframe_to_csv(frame, path):
    temp_path = path.with_name(f"{path.stem}.tmp.csv")
    frame.to_csv(temp_path, index=False, encoding="utf-8", lineterminator="\n")
    os.replace(temp_path, path)


validation_table = pd.concat([fifteen_validation, six_validation], ignore_index=True)
reconciliation_table = pd.concat([fifteen_reconciliation, six_reconciliation], ignore_index=True)
merge_log_table = pd.concat([fifteen_minute_merge_log, six_hour_merge_log], ignore_index=True)

report_tables = {
    "00_environment.csv": environment_table,
    "00_source_header_audit.csv": source_header_audit,
    "00_merge_log.csv": merge_log_table,
    "00_merge_validation.csv": validation_table,
    "00_source_reconciliation.csv": reconciliation_table,
    "00_household_reconciliation.csv": household_reconciliation,
    "00_six_hour_time_duplicate_summary.csv": six_hour_time_and_duplicate_summary,
    "00_six_hour_coverage_summary.csv": six_hour_coverage_summary,
    "00_six_hour_missingness.csv": six_hour_missing_table,
    "00_six_hour_numeric_audit.csv": six_hour_numeric_table,
    "00_six_hour_metadata_stability.csv": metadata_stability_table,
}

if fifteen_minute_raw_audit is not None:
    report_tables.update(
        {
            "00_15minute_source_audit.csv": fifteen_minute_source_audit,
            "00_15minute_record_types.csv": fifteen_minute_record_types,
            "00_15minute_key_missingness.csv": fifteen_minute_missing,
            "00_15minute_value_summary.csv": fifteen_minute_value_summary,
            "00_15minute_value_quantiles.csv": fifteen_minute_value_quantiles.rename_axis("quantile").reset_index(),
            "00_15minute_duplicate_screen.csv": fifteen_minute_duplicate_screen,
            "00_15minute_coverage_summary.csv": fifteen_minute_coverage_summary,
            "00_15minute_coverage_distribution.csv": fifteen_minute_coverage_distribution,
        }
    )
if six_hour_raw_audit is not None:
    report_tables["00_6hour_source_audit.csv"] = six_hour_source_audit

for filename, frame in report_tables.items():
    atomic_dataframe_to_csv(frame, REPORT_TABLE_DIR / filename)

dtype_contract_payload = {
    "contract_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "csv_reading_rule": "Read canonical merged fields as strings; explicitly parse numeric and datetime fields in Notebook 01.",
    "datasets": CSV_DTYPE_CONTRACT,
}
DTYPE_CONTRACT_PATH.write_text(
    json.dumps(dtype_contract_payload, indent=2), encoding="utf-8"
)

manifest = {
    "manifest_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "notebooks/00_environment_and_data_audit.ipynb",
    "execution_mode": MERGE_EXECUTION_MODE,
    "seed": SEED,
    "source_files": source_header_audit[
        ["dataset", "source_file", "size_bytes", "modified_utc"]
    ].to_dict(orient="records"),
    "outputs": [fifteen_signature, six_signature],
    "dtype_contract": str(DTYPE_CONTRACT_PATH.relative_to(PROJECT_ROOT)),
    "all_validations_passed": bool(validation_table["passed"].all()),
}
MERGE_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

created_dataset_paths = [FIFTEEN_MINUTE_MERGED_PATH, SIX_HOUR_MERGED_PATH, *[REPORT_TABLE_DIR / name for name in report_tables]]
non_csv_datasets = [str(path) for path in created_dataset_paths if path.suffix.lower() != ".csv"]
if non_csv_datasets:
    raise AssertionError(f"Non-CSV dataset outputs detected: {non_csv_datasets}")
if not validation_table["passed"].all():
    raise AssertionError("One or more merge validation checks failed")

# Save a compact, data-derived population-flow summary for reporting.
# Household-date gaps are counted from raw LP keys; the cause of a gap is not inferred.
lp_household_day_counts = fifteen_minute_raw_audit["lp_household_day_counts"]
lp_household_date_keys = list(lp_household_day_counts)
lp_dates_by_household = defaultdict(lambda: {"first": None, "last": None})
lp_dates = set()
for household_id, date_value in lp_household_date_keys:
    observed_date = pd.Timestamp(date_value).normalize()
    lp_dates.add(observed_date)
    bounds = lp_dates_by_household[str(household_id)]
    if bounds["first"] is None or observed_date < bounds["first"]:
        bounds["first"] = observed_date
    if bounds["last"] is None or observed_date > bounds["last"]:
        bounds["last"] = observed_date

lp_calendar_span_days = sum(
    (bounds["last"] - bounds["first"]).days + 1
    for bounds in lp_dates_by_household.values()
)
lp_first_date = min(lp_dates)
lp_last_date = max(lp_dates)
lp_global_calendar_dates = pd.date_range(lp_first_date, lp_last_date, freq="D")
lp_global_dates_without_rows = len(set(lp_global_calendar_dates) - lp_dates)
record_type_rows = {
    str(row.obis): int(row.rows)
    for row in fifteen_minute_record_types.itertuples(index=False)
}
raw_15min_rows = int(fifteen_minute_merge_log["rows_written"].sum())
lp_rows = int(record_type_rows.get("LP", 0))
dbe_rows = int(record_type_rows.get("DBE", 0))
other_record_type_rows = raw_15min_rows - lp_rows - dbe_rows
raw_date_start = pd.to_datetime(fifteen_minute_source_audit["min_date"]).min()
raw_date_end = pd.to_datetime(fifteen_minute_source_audit["max_date"]).max()

data_flow_summary = {
    "summary_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "notebooks/00_environment_and_data_audit.ipynb",
    "source_files": [path.name for path in FIFTEEN_MINUTE_SOURCE_FILES],
    "raw_15min_rows": raw_15min_rows,
    "lp_rows": lp_rows,
    "dbe_rows": dbe_rows,
    "other_record_type_rows": int(other_record_type_rows),
    "lp_households": len(lp_dates_by_household),
    "lp_valid_avgimportkw_intervals": int(
        fifteen_minute_raw_audit["lp_avgimportkw_counts"]["valid_finite_nonnegative"]
    ),
    "lp_missing_avgimportkw_intervals": int(
        fifteen_minute_raw_audit["lp_avgimportkw_counts"]["missing"]
    ),
    "lp_invalid_nonmissing_avgimportkw_intervals": int(
        fifteen_minute_raw_audit["lp_avgimportkw_counts"]["non_numeric"]
        + fifteen_minute_raw_audit["lp_avgimportkw_counts"]["non_finite"]
        + fifteen_minute_raw_audit["lp_avgimportkw_counts"]["negative"]
    ),
    "lp_avgimportkw_validity_definition": (
        "Valid means the LP avgimportkw value parses as finite numeric and is nonnegative. "
        "Timestamp validity, duplicate resolution, and daily-aggregation eligibility are "
        "assessed in Notebook 01."
    ),
    "date_start": raw_date_start.date().isoformat(),
    "date_end": raw_date_end.date().isoformat(),
    "lp_observed_household_days": len(lp_household_date_keys),
    "lp_household_calendar_days_between_first_and_last": int(lp_calendar_span_days),
    "lp_empty_household_days_within_first_last_span": int(
        lp_calendar_span_days - len(lp_household_date_keys)
    ),
    "lp_global_dates_without_any_rows_within_date_range": int(
        lp_global_dates_without_rows
    ),
    "gap_count_interpretation": (
        "Household-date keys absent from the supplied LP records within each household's "
        "first-to-last observed date; this does not establish why the records are absent."
    ),
    "merge_validations_passed": bool(validation_table["passed"].all()),
}

if raw_15min_rows != sum(int(value) for value in record_type_rows.values()):
    raise AssertionError("15-minute record-type counts do not reconcile to raw rows")
if lp_rows != sum(int(value) for value in lp_household_day_counts.values()):
    raise AssertionError("Raw LP household-day counts do not reconcile to LP rows")
lp_import_counts = fifteen_minute_raw_audit["lp_avgimportkw_counts"]
if lp_rows != int(lp_import_counts["lp_rows"]):
    raise AssertionError("LP avgimportkw status counts do not reconcile to LP rows")
lp_import_status_total = sum(
    int(lp_import_counts[key])
    for key in ["valid_finite_nonnegative", "missing", "non_numeric", "non_finite", "negative"]
)
if lp_rows != lp_import_status_total:
    raise AssertionError("LP avgimportkw validity categories do not reconcile to LP rows")
if data_flow_summary["lp_empty_household_days_within_first_last_span"] < 0:
    raise AssertionError("LP observed household-days exceed their calendar spans")

DATA_AUDIT_SUMMARY_PATH = REPORT_TABLE_DIR / "00_data_audit_summary.json"
DATA_AUDIT_SUMMARY_PATH.write_text(
    json.dumps(data_flow_summary, indent=2), encoding="utf-8"
)
summary_readback = json.loads(DATA_AUDIT_SUMMARY_PATH.read_text(encoding="utf-8"))
if summary_readback != data_flow_summary:
    raise AssertionError("Dataset-flow summary JSON read-back did not match")
display(pd.DataFrame([data_flow_summary]))

population_flow_table = pd.DataFrame(
    [
        {"measure": "Raw 15-minute records", "value": raw_15min_rows, "definition": "All rows in the supplied 15-minute source files."},
        {"measure": "LP records", "value": lp_rows, "definition": "Rows classified as OBIS LP."},
        {"measure": "LP households", "value": len(lp_dates_by_household), "definition": "Distinct LP household IDs with dated records."},
        {"measure": "Valid avgimportkw intervals", "value": int(lp_import_counts["valid_finite_nonnegative"]), "definition": "LP avgimportkw parses as finite numeric and is nonnegative; later timestamp/duplicate rules are not applied here."},
        {"measure": "Missing avgimportkw intervals", "value": int(lp_import_counts["missing"]), "definition": "LP rows with an empty/missing avgimportkw value."},
        {"measure": "Invalid nonmissing avgimportkw intervals", "value": int(data_flow_summary["lp_invalid_nonmissing_avgimportkw_intervals"]), "definition": "LP rows with nonnumeric, nonfinite, or negative avgimportkw values."},
        {"measure": "Raw 15-minute date range", "value": f"{data_flow_summary['date_start']} to {data_flow_summary['date_end']}", "definition": "Minimum to maximum parsed date across the supplied 15-minute source files."},
    ]
)
POPULATION_FLOW_TABLE_PATH = REPORT_TABLE_DIR / "00_research_population_flow.csv"
atomic_dataframe_to_csv(population_flow_table, POPULATION_FLOW_TABLE_PATH)
report_tables[POPULATION_FLOW_TABLE_PATH.name] = population_flow_table
created_dataset_paths.append(POPULATION_FLOW_TABLE_PATH)
if any(path.suffix.lower() != ".csv" for path in created_dataset_paths):
    raise AssertionError("Population-flow report must remain a CSV output")
flow_table_readback = pd.read_csv(POPULATION_FLOW_TABLE_PATH)
if not flow_table_readback["measure"].equals(population_flow_table["measure"]):
    raise AssertionError("Population-flow table read-back did not preserve its measures")

saved_artifacts = pd.DataFrame(
    [
        {"artifact": str(path.relative_to(PROJECT_ROOT)), "bytes": path.stat().st_size}
        for path in [
            FIFTEEN_MINUTE_MERGED_PATH, SIX_HOUR_MERGED_PATH,
            MERGE_MANIFEST_PATH, DTYPE_CONTRACT_PATH,
            DATA_AUDIT_SUMMARY_PATH, POPULATION_FLOW_TABLE_PATH,
        ]
    ]
)
display(saved_artifacts)
print(f"Saved {len(report_tables)} report tables to {REPORT_TABLE_DIR}")
print(f"Saved dataset-flow summary to {DATA_AUDIT_SUMMARY_PATH}")
print("Notebook 00 completed: every validation passed and every tabular output is CSV.")

## Completion interpretation

A successful execution establishes the following boundaries:

- the raw files were schema-compatible at the time of execution;
- every raw row was preserved in the correct canonical CSV;
- provenance is complete, sequential, and unique;
- the merged files can be parsed back using the declared CSV contract;
- high demand values were retained for later outlier analysis;
- six-hour network metadata remains separate and must later be joined by
  household and valid time, not by a single household-wide value.

It does **not** declare rows clean or model-ready. Duplicate resolution,
timestamp/timezone semantics, LP policy, valid-day completeness, missing-value
treatment, outlier analysis, and the time-valid metadata join are Notebook 01
responsibilities.